# Pacer Dashboard — Streamlit + MQTT (EMQX) di Colab

Jalankan sel dari atas ke bawah. Hasil akhir: dashboard Streamlit dengan URL publik, yang **auto-refresh tiap 2 detik** di tab Live Dashboard.

**Perubahan utama dibanding versi lama**
- Live Dashboard memakai `st.fragment(run_every=2)`: hanya tab live yang di-refresh, form di tab Rencana tidak terganggu.
- Event disimpan di store global yang diproteksi lock (tidak ada lagi balapan `for ... / clear()`, aman untuk banyak tab, dan tidak hilang saat F5).
- Klien MQTT: `connect_async` + reconnect otomatis, `client_id` acak, klien lama dimatikan kalau cache ter-reset (tidak ada klien "hantu").
- Status koneksi, status subscribe (SUBACK), waktu pesan terakhir, dan error terakhir tampil di dashboard.
- Tombol **Kirim jadwal** hanya sukses kalau broker mengkonfirmasi (sebelumnya selalu menampilkan sukses).
- Password tidak di-hardcode.

> Ganti password MQTT lama (`rawr`) di EMQX, karena sudah tersebar di chat dan kode.

## 1. Install dependensi

In [ ]:
!pip -q install "streamlit>=1.37" "paho-mqtt>=2.0" plotly pandas certifi

import importlib.metadata as md
v = md.version("paho-mqtt")
assert int(v.split(".")[0]) >= 2, f"paho-mqtt {v} terlalu lama. Runtime > Restart session, lalu jalankan ulang sel ini."
print("paho-mqtt", v, "| streamlit", md.version("streamlit"))

## 2. Konfigurasi
Password diminta lewat `getpass` (tidak tersimpan di notebook). Kalau broker/user/topik berbeda, ubah nilai di bawah.

In [ ]:
import os, getpass

os.environ["MQTT_BROKER"]     = "y78c1fd2.ala.asia-southeast1.emqxsl.com"
os.environ["MQTT_PORT"]       = "8883"
os.environ["MQTT_USER"]       = "ORCA-CobSIC-Admin4"
os.environ["TOPIC_SCHEDULE"]  = "sic8/ORCA-CobSIC/schedule"
os.environ["TOPIC_EVENT"]     = "sic8/ORCA-CobSIC/event"
os.environ["MQTT_TLS"]        = "1"

if not os.environ.get("MQTT_PASS"):
    os.environ["MQTT_PASS"] = getpass.getpass("MQTT password: ")
print("Konfigurasi siap untuk user:", os.environ["MQTT_USER"])

## 3. Tulis `app.py`

In [ ]:
%%writefile app.py
import json
import os
import threading
import uuid
from datetime import datetime

import paho.mqtt.client as mqtt
import pandas as pd
import plotly.express as px
import streamlit as st

# set_page_config harus jadi perintah Streamlit pertama
st.set_page_config(page_title="Pacer Dashboard", layout="wide")


# ==========================================
# KONFIGURASI (env var -> st.secrets -> default)
# Password TIDAK di-hardcode: isi lewat env var MQTT_PASS atau .streamlit/secrets.toml
# ==========================================
def cfg(name, default=None):
    value = os.environ.get(name)
    if value:
        return value
    try:
        return st.secrets[name]
    except Exception:
        return default


MQTT_BROKER = cfg("MQTT_BROKER", "y78c1fd2.ala.asia-southeast1.emqxsl.com")
MQTT_PORT = int(cfg("MQTT_PORT", 8883))
MQTT_USER = cfg("MQTT_USER", "ORCA-CobSIC-Admin4")
MQTT_PASS = cfg("MQTT_PASS")
MQTT_TLS = str(cfg("MQTT_TLS", "1")).lower() not in ("0", "false", "no")
TOPIC_SCHEDULE = cfg("TOPIC_SCHEDULE", "sic8/ORCA-CobSIC/schedule")
TOPIC_EVENT = cfg("TOPIC_EVENT", "sic8/ORCA-CobSIC/event")

MAX_TASKS = 6      # batas tugas per hari (kapasitas memori ESP32)
MAX_EVENTS = 500   # batas riwayat event di memori server
STATUS_EVENTS = ["STARTED", "DELAYED", "MISSED", "DONE", "REALLOCATED", "BREAK"]

if not MQTT_PASS:
    st.error(
        "MQTT_PASS belum diisi. Set environment variable `MQTT_PASS` "
        "(atau `.streamlit/secrets.toml`) lalu jalankan ulang aplikasi."
    )
    st.stop()


# ==========================================
# STORE GLOBAL (satu per proses server, dipakai semua sesi browser)
# Thread MQTT menulis ke sini; UI hanya membaca. Tidak ada pemanggilan Streamlit
# dari thread MQTT.
# ==========================================
@st.cache_resource
def get_store():
    return {
        "lock": threading.Lock(),
        "events": [],
        "subscribed": None,   # None = belum tahu, True/False = hasil SUBACK
        "last_msg": None,
        "bad_payloads": 0,
        "last_error": None,
        "seen_retained": set(),
        "client": None,
    }


# ==========================================
# KLIEN MQTT (background thread, dibuat sekali per proses)
# ==========================================
@st.cache_resource
def init_mqtt():
    s = get_store()

    # Kalau cache ter-reset (mis. file diedit), matikan klien lama agar tidak jadi "hantu"
    old = s.get("client")
    if old is not None:
        try:
            old.disconnect()
            old.loop_stop()
        except Exception:
            pass

    def on_connect(client, userdata, flags, reason_code, properties):
        if reason_code.is_failure:
            with s["lock"]:
                s["last_error"] = f"Koneksi ditolak broker: {reason_code}"
            return
        with s["lock"]:
            s["last_error"] = None
        # Subscribe di sini supaya otomatis subscribe ulang setelah reconnect
        client.subscribe(TOPIC_EVENT, qos=1)

    def on_subscribe(client, userdata, mid, reason_code_list, properties):
        ok = all(not rc.is_failure for rc in reason_code_list)
        with s["lock"]:
            s["subscribed"] = ok
            if not ok:
                s["last_error"] = (
                    f"Subscribe ke '{TOPIC_EVENT}' ditolak broker "
                    f"({list(reason_code_list)}). Cek ACL user di EMQX."
                )

    def on_disconnect(client, userdata, disconnect_flags, reason_code, properties):
        with s["lock"]:
            s["subscribed"] = None
            # jangan timpa pesan error yang lebih spesifik (mis. "Not authorized")
            if reason_code.is_failure and s["last_error"] is None:
                s["last_error"] = f"Terputus dari broker: {reason_code}"

    def on_connect_fail(client, userdata):
        with s["lock"]:
            s["last_error"] = (
                "Gagal menyambung ke broker (cek host/port/TLS/internet). Mencoba lagi otomatis..."
            )

    def on_message(client, userdata, msg):
        # JANGAN sampai ada exception lolos: exception di callback bisa mematikan thread MQTT
        try:
            try:
                data = json.loads(msg.payload.decode("utf-8"))
                if not isinstance(data, dict):
                    raise ValueError("payload bukan JSON object")
            except Exception as e:
                with s["lock"]:
                    s["bad_payloads"] += 1
                    s["last_error"] = (
                        f"Payload tidak valid di '{msg.topic}': {bytes(msg.payload)[:80]!r} ({e})"
                    )
                print(f"[MQTT] payload tidak valid: {e}", flush=True)
                return

            with s["lock"]:
                if msg.retain:
                    # Pesan retained dikirim ulang tiap reconnect -> cegah duplikat
                    key = (msg.topic, bytes(msg.payload))
                    if key in s["seen_retained"]:
                        return
                    s["seen_retained"].add(key)
                s["events"].append(data)
                del s["events"][:-MAX_EVENTS]
                s["last_msg"] = datetime.now()
        except Exception as e:  # pragma: no cover
            print(f"[MQTT] error di on_message: {e}", flush=True)

    # client_id acak: dua instance dashboard (mis. laptop + Colab) tidak saling menendang
    client = mqtt.Client(
        mqtt.CallbackAPIVersion.VERSION2,
        client_id=f"pacer-dash-{uuid.uuid4().hex[:8]}",
    )
    if MQTT_TLS:
        try:
            import certifi

            client.tls_set(ca_certs=certifi.where())
        except ImportError:
            client.tls_set()
    client.username_pw_set(MQTT_USER, MQTT_PASS)
    client.reconnect_delay_set(min_delay=1, max_delay=30)
    client.on_connect = on_connect
    client.on_subscribe = on_subscribe
    client.on_disconnect = on_disconnect
    client.on_connect_fail = on_connect_fail
    client.on_message = on_message

    # connect_async + loop_start: tidak memblokir/crash kalau broker sedang tidak bisa dijangkau,
    # dan otomatis mencoba lagi
    client.connect_async(MQTT_BROKER, MQTT_PORT, keepalive=30)
    client.loop_start()
    s["client"] = client
    return client


store = get_store()
mqtt_client = init_mqtt()


def publish_checked(topic, payload, retain=False, timeout=5):
    """Publish QoS 1 dan tunggu konfirmasi broker. Return (ok, pesan_error)."""
    if not mqtt_client.is_connected():
        return False, "Dashboard belum terhubung ke broker MQTT."
    try:
        info = mqtt_client.publish(topic, json.dumps(payload), qos=1, retain=retain)
        info.wait_for_publish(timeout=timeout)
    except (ValueError, RuntimeError) as e:
        return False, f"Publish gagal: {e}"
    if not info.is_published():
        return False, f"Broker tidak mengkonfirmasi dalam {timeout} detik (cek ACL publish)."
    return True, None


# ==========================================
# SIDEBAR: info koneksi & alat uji
# ==========================================
with st.sidebar:
    st.markdown("### 🔧 Alat Uji")
    st.caption(f"Broker: `{MQTT_BROKER}:{MQTT_PORT}` ({'TLS' if MQTT_TLS else 'tanpa TLS'})")
    st.caption(f"Subscribe: `{TOPIC_EVENT}`")
    st.caption(f"Publish jadwal: `{TOPIC_SCHEDULE}`")
    if st.button("🧪 Kirim event tes"):
        ok, err = publish_checked(
            TOPIC_EVENT,
            {"event": "STARTED", "task": "TES-DASHBOARD", "sim": datetime.now().strftime("%H:%M")},
        )
        if ok:
            st.success("Terkirim. Kalau pipeline sehat, event muncul di Live Dashboard dalam ±2 detik.")
        else:
            st.error(err)

# --- HEADER APLIKASI ---
st.markdown("<h1 style='text-align: center; color: #4CAF50;'>Pacer 🎯</h1>", unsafe_allow_html=True)
st.markdown(
    "<h4 style='text-align: center;'>Smart Task Committer - Bikin Kamu Makin Produktif! 🚀</h4>",
    unsafe_allow_html=True,
)
st.divider()

if "tasks" not in st.session_state:
    st.session_state.tasks = []

tab1, tab2 = st.tabs(["📋 Rencana Harian", "📡 Live Dashboard"])

# ==========================================
# TAB 1: PLANNER
# ==========================================
with tab1:
    st.markdown("### 🗓️ Yuk, Susun Jadwal Hari Ini!")
    st.info(
        "💡 **Tips:** Pacer bakal ngingetin kamu lewat layar OLED ESP32. "
        "Jangan lupa masukin waktu istirahat juga ya!",
        icon="ℹ️",
    )

    day_name = st.selectbox("Pilih Hari", ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])

    with st.form("task_form", clear_on_submit=True):
        col1, col2 = st.columns(2)
        with col1:
            task_name = st.text_input(
                "📌 Nama Tugas (Maks 20 Karakter)", max_chars=20, placeholder="Contoh: Ngerjain modul Python"
            )
            tier = st.selectbox(
                "⭐ Kategori (Tier)",
                [1, 2, 3],
                help="T1: Super Penting (Deadline), T2: Growth (Bisa digeser), T3: Istirahat/Me-Time",
            )
        with col2:
            start_time = st.time_input("⏰ Mulai Jam Berapa?")
            end_time = st.time_input("⏳ Target Selesai Jam Berapa?")

        st.markdown("---")
        if_then = st.text_input(
            "🔥 Kalimat Komitmen (Kalau..., maka...)",
            max_chars=60,
            placeholder="Contoh: Kalau aku malah scroll TikTok, marahin aja!",
            help="Kalimat ini bakal muncul di layar OLED ESP32 buat nyemangatin (atau ngingetin) kamu.",
        )

        submitted = st.form_submit_button("➕ Tambah ke Jadwal", type="primary")

        if submitted:
            if len(st.session_state.tasks) >= MAX_TASKS:
                st.error(f"Batas maksimal {MAX_TASKS} tugas per hari sesuai kapasitas memori ESP32!")
            elif start_time >= end_time:
                st.error("Waktu selesai harus setelah waktu mulai dong!")
            elif not task_name.strip() or not if_then.strip():
                st.error("Nama tugas dan kalimat komitmen jangan dikosongin ya.")
            else:
                st.session_state.tasks.append(
                    {
                        "name": task_name.strip(),
                        "tier": tier,
                        "start": start_time.strftime("%H:%M"),
                        "end": end_time.strftime("%H:%M"),
                        "if_then": if_then.strip(),
                    }
                )
                st.toast(f"Mantap! Tugas '{task_name.strip()}' berhasil ditambah. 🎉", icon="✅")

    if st.session_state.tasks:
        st.subheader("Daftar Tugas Harian Kamu")
        st.table(pd.DataFrame(st.session_state.tasks))

        col_clear, col_send = st.columns([1, 1])
        if col_clear.button("🗑️ Hapus Semua Tugas"):
            st.session_state.tasks = []
            st.rerun()

        if col_send.button("🚀 Kirim Jadwal ke ESP32", type="primary"):
            ok, err = publish_checked(
                TOPIC_SCHEDULE,
                {"day": day_name, "tasks": st.session_state.tasks},
                retain=True,
            )
            if ok:
                st.success("Wusss! 🌬️ Jadwal berhasil dikirim ke Pacer ESP32!")
            else:
                st.error(f"Jadwal TIDAK terkirim. {err}")


# ==========================================
# TAB 2: LIVE DASHBOARD
# Hanya fragment ini yang dijalankan ulang tiap 2 detik: form di Tab 1 tidak terganggu.
# ==========================================
@st.fragment(run_every=2)
def live_view():
    if st.button("🧹 Bersihkan log"):
        with store["lock"]:
            store["events"].clear()

    with store["lock"]:
        events = list(store["events"])
        subscribed = store["subscribed"]
        last_msg = store["last_msg"]
        bad = store["bad_payloads"]
        last_error = store["last_error"]

    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Broker", "🟢 Terhubung" if mqtt_client.is_connected() else "🔴 Terputus")
    c2.metric("Subscribe", {True: "✅ Aktif", False: "❌ Ditolak", None: "⏳ Menunggu"}[subscribed])
    c3.metric("Pesan terakhir", last_msg.strftime("%H:%M:%S") if last_msg else "belum ada")
    c4.metric("Event / Payload rusak", f"{len(events)} / {bad}")
    if last_error:
        st.warning(last_error)

    summaries = [e for e in events if e.get("event") == "DAY_SUMMARY"]
    if summaries:
        ds = summaries[-1]
        st.subheader("🏆 Daily Grade")
        m1, m2, m3 = st.columns(3)
        m1.metric("Skor Harian", f"{ds.get('score', 0)}")
        m2.metric("Grade Akhir", f"{ds.get('grade', '-')}")
        m3.metric("Poin Dasar", f"{ds.get('num', 0)} / {ds.get('den', 0)}")

    st.divider()

    if not events:
        st.info("Belum ada event yang diterima dari Pacer. Menunggu data...")
        return

    col_log, col_chart = st.columns([2, 3])

    with col_log:
        st.subheader("Log Kejadian")
        for evt in reversed(events[-100:]):
            event_type = evt.get("event")
            task = evt.get("task", "Unknown")
            sim_time = evt.get("sim", "--:--")
            if event_type == "DAY_SUMMARY":
                st.success(f"🏁 DAY_SUMMARY: Score {evt.get('score')}, Grade {evt.get('grade')}")
            elif event_type == "SCHEDULE_LOADED":
                st.info(f"📥 Jadwal dimuat di perangkat (Hari: {evt.get('day')})")
            elif event_type == "STARTED":
                st.markdown(f"**{sim_time}** - 🟢 `{task}` **MULAI**")
            elif event_type == "DELAYED":
                st.markdown(f"**{sim_time}** - 🟠 `{task}` **DELAY** ({evt.get('min')} min)")
            elif event_type == "MISSED":
                st.markdown(f"**{sim_time}** - 🔴 `{task}` **TERLEWAT (MISSED)**")
            elif event_type == "REALLOCATED":
                st.markdown(f"**{sim_time}** - 🔵 `{task}` **DIREALOKASI**")
            elif event_type == "BREAK":
                st.markdown(f"**{sim_time}** - ☕ `{task}` **ISTIRAHAT** ({evt.get('min')} min)")
            elif event_type == "DONE":
                st.markdown(f"**{sim_time}** - ✅ `{task}` **SELESAI**")
            else:
                # Event tak dikenal (mis. nama key beda) tetap ditampilkan agar mudah di-debug
                st.text(f"❓ {json.dumps(evt, ensure_ascii=False)}")

    with col_chart:
        st.subheader("Status Tugas")
        df_events = pd.DataFrame(events)
        if {"event", "task"} <= set(df_events.columns):
            status_df = df_events[df_events["event"].isin(STATUS_EVENTS) & df_events["task"].notna()]
            if not status_df.empty:
                fig = px.histogram(
                    status_df,
                    x="task",
                    color="event",
                    title="Statistik Status per Tugas",
                    barmode="group",
                )
                st.plotly_chart(fig)
            else:
                st.caption("Belum ada event berstatus tugas untuk digrafikkan.")
        else:
            st.caption("Belum ada event berstatus tugas untuk digrafikkan.")


with tab2:
    st.header("Live Event Feed")
    live_view()

## 4. Cek koneksi ke broker *sebelum* menjalankan Streamlit
Sel ini memeriksa: TLS, username/password, izin subscribe, izin publish, dan perjalanan pesan (publish lalu diterima kembali).
Kalau sel ini gagal, masalahnya di broker/kredensial/ACL, bukan di Streamlit.

In [ ]:
import os, json, time, ssl, threading, uuid
import certifi
import paho.mqtt.client as mqtt

B, P = os.environ["MQTT_BROKER"], int(os.environ["MQTT_PORT"])
T = os.environ["TOPIC_EVENT"]
res = {"connect": None, "suba": None, "echo": False}
done = threading.Event()
marker = uuid.uuid4().hex[:8]

def on_connect(c, u, f, rc, p):
    res["connect"] = str(rc)
    if not rc.is_failure:
        c.subscribe(T, qos=1)
    else:
        done.set()

def on_subscribe(c, u, mid, rcs, p):
    res["suba"] = [str(x) for x in rcs]
    if all(not x.is_failure for x in rcs):
        c.publish(T, json.dumps({"event": "STARTED", "task": f"CEK-{marker}", "sim": "00:00"}), qos=1)
    else:
        done.set()

def on_message(c, u, m):
    if marker in m.payload.decode("utf-8", "ignore"):
        res["echo"] = True
        done.set()

cl = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id=f"cek-{marker}")
cl.tls_set(ca_certs=certifi.where())
cl.username_pw_set(os.environ["MQTT_USER"], os.environ["MQTT_PASS"])
cl.on_connect, cl.on_subscribe, cl.on_message = on_connect, on_subscribe, on_message
try:
    cl.connect(B, P, keepalive=30)
    cl.loop_start()
    done.wait(15)
except Exception as e:
    print("GAGAL menyambung:", type(e).__name__, e)
finally:
    cl.loop_stop(); cl.disconnect()

print("CONNACK :", res["connect"])
print("SUBACK  :", res["suba"])
print("Echo    :", "OK" if res["echo"] else "TIDAK DITERIMA")
if res["echo"]:
    print("\nBroker, kredensial, ACL subscribe dan publish semuanya beres. Lanjut ke langkah 5.")
elif res["connect"] and "Not authorized" in res["connect"]:
    print("\nUsername/password salah, atau user belum dibuat di EMQX (Access Control > Authentication).")
elif res["suba"] and any("Not authorized" in x or "failure" in x.lower() for x in res["suba"]):
    print("\nSubscribe ditolak: cek Authorization/ACL user di EMQX untuk topik", T)
else:
    print("\nTidak ada echo: cek ACL publish, host/port, atau koneksi internet runtime Colab.")

## 5. Jalankan Streamlit di background

In [ ]:
import subprocess, time, urllib.request, os

# hentikan instance lama kalau sel ini dijalankan ulang
!pkill -f "streamlit run" 2>/dev/null; sleep 1

log = open("streamlit.log", "w")
proc = subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port", "8501", "--server.headless", "true",
     "--server.enableCORS", "false", "--server.enableXsrfProtection", "false",
     "--browser.gatherUsageStats", "false"],
    stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy(),
)

ok = False
for _ in range(40):
    try:
        if urllib.request.urlopen("http://localhost:8501/_stcore/health", timeout=2).read() == b"ok":
            ok = True; break
    except Exception:
        time.sleep(1)
print("Streamlit hidup di port 8501" if ok else "Streamlit belum siap, lihat streamlit.log:")
if not ok:
    print(open("streamlit.log").read()[-2000:])

## 6. Buka lewat URL publik (Cloudflare quick tunnel, tanpa akun)
Cari alamat `https://....trycloudflare.com` di output, lalu buka. **Siapa pun yang punya URL ini bisa membuka dashboard dan mengirim jadwal**, jadi jangan disebar sembarangan. URL berganti setiap sel ini dijalankan.

In [ ]:
import subprocess, re, time, os, urllib.request

!pkill -f cloudflared 2>/dev/null
if not os.path.exists("cloudflared"):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "cloudflared")
    os.chmod("cloudflared", 0o755)

tun_log = open("tunnel.log", "w")
tun = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
                       stdout=tun_log, stderr=subprocess.STDOUT)
url = None
for _ in range(40):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        url = m.group(0); break
    time.sleep(1)
print("URL dashboard:", url or "belum muncul, lihat tunnel.log")

## 7. Simulasi perangkat (tanpa Wokwi)
Mengirim urutan event palsu ke topik event, supaya seluruh UI bisa dites. Kalau Live Dashboard terisi, bagian Streamlit-nya sehat, dan sisa masalah (kalau ada) ada di sisi Wokwi/ESP32.

In [ ]:
import os, json, time, certifi
import paho.mqtt.client as mqtt

T = os.environ["TOPIC_EVENT"]
events = [
    {"event": "SCHEDULE_LOADED", "day": "Mon"},
    {"event": "STARTED", "task": "Belajar", "sim": "08:00"},
    {"event": "DELAYED", "task": "Belajar", "sim": "08:05", "min": 10},
    {"event": "BREAK", "task": "Belajar", "sim": "08:40", "min": 15},
    {"event": "DONE", "task": "Belajar", "sim": "09:30"},
    {"event": "STARTED", "task": "Gym", "sim": "10:00"},
    {"event": "MISSED", "task": "Gym", "sim": "10:05"},
    {"event": "REALLOCATED", "task": "Gym", "sim": "10:06"},
    {"event": "DAY_SUMMARY", "score": 82, "grade": "B", "num": 5, "den": 6},
]
c = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id="sim-device")
c.tls_set(ca_certs=certifi.where())
c.username_pw_set(os.environ["MQTT_USER"], os.environ["MQTT_PASS"])
c.connect(os.environ["MQTT_BROKER"], int(os.environ["MQTT_PORT"]), keepalive=30)
c.loop_start()
for e in events:
    c.publish(T, json.dumps(e), qos=1).wait_for_publish(5)
    print("dikirim:", e)
    time.sleep(1)
c.loop_stop(); c.disconnect()

## 8. Matikan semuanya

In [ ]:
!pkill -f "streamlit run" 2>/dev/null
!pkill -f cloudflared 2>/dev/null
print("Streamlit dan tunnel dimatikan.")

## Catatan untuk sisi Wokwi / ESP32
- Event harus dipublish ke **persis** topik `sic8/ORCA-CobSIC/event` sebagai **JSON object** (bukan teks biasa atau array) dengan key `event` (`STARTED`, `DELAYED`, `MISSED`, `REALLOCATED`, `BREAK`, `DONE`, `SCHEDULE_LOADED`, `DAY_SUMMARY`). Event dengan key lain tetap tampil sebagai baris `❓` supaya mudah dilacak.
- Pakai `client_id` yang **berbeda** untuk ESP32 dan dashboard. Dua klien dengan ID sama saling menendang.
- Dashboard hanya menerima event yang terbit **setelah** ia subscribe, kecuali event itu `retain`. Nyalakan dashboard dulu sebelum menjalankan simulasi Wokwi.
- Jadwal dikirim ke `sic8/ORCA-CobSIC/schedule` dengan `retain=True`, jadi ESP32 yang subscribe belakangan tetap menerima jadwal terakhir.
- Runtime Colab gratis terputus saat idle lama atau setelah beberapa jam. Kalau tunnel mati, jalankan ulang langkah 5 dan 6.